# Experiment

The 3 layer version of the offense defense model had high variance. I wonder what the performance could look like on the 2 layer model instead. This should no longer have high variance

## Results

The model got the following score:

```
Benchmark Score Accuracy: 0.18
Benchmark Win Accuracy  : 0.7542
Training Score Accuracy : 0.12219451371571072
Training Win Accuracy   : 0.4613466334164589
CV Set Score Accuracy   : 0.12244897959183673
CV Set Win Accuracy     : 0.5

--------------------------------------------------------------

    Precision and recall
 1-0   TP:5  FP:19  FN:6: Prec:0.2083  Rec:0.4545  F1:14.0000  #Pred: 24  Sample:11
 3-0   TP:0  FP:10  FN:10: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 10  Sample:10
 0-1   TP:0  FP:6  FN:10: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 6  Sample:10
 0-2   TP:7  FP:29  FN:3: Prec:0.1944  Rec:0.7000  F1:13.1429  #Pred: 36  Sample:10
 0-4   TP:0  FP:0  FN:10: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:10
 4-0   TP:0  FP:0  FN:8: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:8
 2-0   TP:0  FP:0  FN:8: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:8
 0-3   TP:0  FP:22  FN:8: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 22  Sample:8
 0-5   TP:0  FP:0  FN:7: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:7
 0-0   TP:0  FP:0  FN:6: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:6
 5-0   TP:0  FP:0  FN:3: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:3
 6-0   TP:0  FP:0  FN:3: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:3
10-10  TP:0  FP:0  FN:1: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:1
 0-6   TP:0  FP:0  FN:1: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:1
 0-9   TP:0  FP:0  FN:1: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:1
 7-0   TP:0  FP:0  FN:1: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:1
```

This performance is definetly better than in the 3 layer model. My suspicion that it was indeed suffering from high variance was correct. Maybe I should add more units in the first layer to see if that makes any difference (first layer had 64 units)

Performance with 80 units was slightly worse. 64 units seems to be a sweet spot

In [20]:
import sys, os
sys.path.insert(0, "/home/roman/Code/AIEngineering/Projects/FIFACompetition/WorldCup2026/experiments")

In [21]:
# Imports
# Allowing notebook to import from components folder
import sys, os
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.insert(0, root_path)
    
import numpy as np
from components.player_modification import arrange_player_details_to_offense_defense
from components.data import load_original_data_with_trimmed_y, collect_player_data_into_sum_concat_teams_form_from_old, split_and_normalize_dataset
from components.constants import alpha
from models.v2 import ModelV2
from components.evaluations import Evaluation

# Loading Data

Load and normalize offense defense data

In [22]:
X, Y = load_original_data_with_trimmed_y()
X_offense_defense = collect_player_data_into_sum_concat_teams_form_from_old(
    Input=X,
    num_player_features=8,
    extract_player_vector=arrange_player_details_to_offense_defense
)

X_train, X_cv, X_test, Y_train, Y_cv, Y_test = split_and_normalize_dataset(
    X=X_offense_defense,
    Y=Y,
    training_set_ratio=0.86,
    cv_test_set_ratio=0.5,
    random_state=121
)

print(X_train.shape, Y_train.shape)
print(X_cv.shape, Y_cv.shape)
print(X_test.shape, Y_test.shape)

(16, 1203) (121, 1203)
(16, 98) (121, 98)
(16, 99) (121, 99)


# Train

Train the model

In [23]:
model_v2 = ModelV2(W1_shape=(64,16), b1_shape=(64,1), W2_shape=(121, 64), b2_shape=(121,1), seed=56)
init_W1, init_b1, init_W2, init_b2 = model_v2.initialize_weights()

# model_v2.gradient_checking(X=X_cv, Y=Y_cv, W1=init_W1, b1=init_b1, W2=init_W2, b2=init_b2) confirmed implementation of model is correct

updated_W1, updated_W2, updated_b1, updated_b2, Js = model_v2.train(
    X=X_train, 
    W1=init_W1,
    b1=init_b1,
    W2=init_W2,
    b2=init_b2,
    Y=Y_train,
    alpha=alpha,
    num_iters=1000
)

Cost at epoch 0/1000 = 4.813503390546367
Cost at epoch 1/1000 = 4.810484104157539
Cost at epoch 2/1000 = 4.807472369520661
Cost at epoch 3/1000 = 4.804467407019627
Cost at epoch 4/1000 = 4.801468535125967
Cost at epoch 5/1000 = 4.798475119876701
Cost at epoch 6/1000 = 4.795486799191973
Cost at epoch 7/1000 = 4.792502318712342
Cost at epoch 8/1000 = 4.789520955863129
Cost at epoch 9/1000 = 4.786541129776208
Cost at epoch 10/1000 = 4.783562384821109
Cost at epoch 11/1000 = 4.780584171153175
Cost at epoch 12/1000 = 4.777606351513226
Cost at epoch 13/1000 = 4.774627491985425
Cost at epoch 14/1000 = 4.7716472932292255
Cost at epoch 15/1000 = 4.768664802923986
Cost at epoch 16/1000 = 4.76567983091379
Cost at epoch 17/1000 = 4.762690805953852
Cost at epoch 18/1000 = 4.759697053097935
Cost at epoch 19/1000 = 4.756697858151239
Cost at epoch 20/1000 = 4.753692457908668
Cost at epoch 21/1000 = 4.750680261831496
Cost at epoch 22/1000 = 4.747659471940471
Cost at epoch 23/1000 = 4.74462997670535
Cos

# Evaluate Model

In [24]:
# Checking if model has high bias, high variance or has reached benchmarks
train_set_pred, _, _, = ModelV2.f_x(X=X_train, W1=updated_W1, b1=updated_b1, W2=updated_W2, b2=updated_b2)
cv_set_pred, _, _, = ModelV2.f_x(X=X_cv, W1=updated_W1, b1=updated_b1, W2=updated_W2, b2=updated_b2)

training_set_accuracy = Evaluation.accuracy_score(train_set_pred, Y_train)
cv_set_accuracy = Evaluation.accuracy_score(cv_set_pred, Y_cv)
training_set_win_pred_accuracy = Evaluation.win_prediction_accuracy(y_pred=train_set_pred, y_label=Y_train)
cv_set_win_pred_accuracy = Evaluation.win_prediction_accuracy(y_pred=cv_set_pred, y_label=Y_cv)

print(f"Benchmark Score Accuracy: 0.18")
print(f"Benchmark Win Accuracy  : 0.7542")
print(f"Training Score Accuracy : {training_set_accuracy}")
print(f"Training Win Accuracy   : {training_set_win_pred_accuracy}")
print(f"CV Set Score Accuracy   : {cv_set_accuracy}")
print(f"CV Set Win Accuracy     : {cv_set_win_pred_accuracy}")

Benchmark Score Accuracy: 0.18
Benchmark Win Accuracy  : 0.7542
Training Score Accuracy : 0.12219451371571072
Training Win Accuracy   : 0.4613466334164589
CV Set Score Accuracy   : 0.12244897959183673
CV Set Win Accuracy     : 0.5


In [25]:
Evaluation.precision_recall(y_pred=cv_set_pred, y_label=Y_cv)

    Precision and recall
 1-0   TP:5  FP:19  FN:6: Prec:0.2083  Rec:0.4545  F1:14.0000  #Pred: 24  Sample:11
 3-0   TP:0  FP:10  FN:10: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 10  Sample:10
 0-1   TP:0  FP:6  FN:10: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 6  Sample:10
 0-2   TP:7  FP:29  FN:3: Prec:0.1944  Rec:0.7000  F1:13.1429  #Pred: 36  Sample:10
 0-4   TP:0  FP:0  FN:10: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:10
 4-0   TP:0  FP:0  FN:8: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:8
 2-0   TP:0  FP:0  FN:8: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:8
 0-3   TP:0  FP:22  FN:8: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 22  Sample:8
 0-5   TP:0  FP:0  FN:7: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:7
 0-0   TP:0  FP:0  FN:6: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:6
 5-0   TP:0  FP:0  FN:3: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample:3
 6-0   TP:0  FP:0  FN:3: Prec:0.0000  Rec:0.0000  F1:0.0000  #Pred: 0  Sample